# Cerco Quickstart

This notebook demonstrates core static analysis APIs on a small source snippet.

In [ ]:
import textwrap

from analysis.capability import analyze_to_dict as capability_to_dict
from analysis.resource import analyze_to_dict as resource_to_dict
from analysis.taint import analyze_to_dict as taint_to_dict
from analysis.decision import ExternalAccessPolicy, SafetyDecisionEngine
from analysis.manifest import ManifestVerificationEngine, generate_manifest_from_source
from analysis.safety_ir import build_safety_ir_from_source
from cfg import build_cfg

In [ ]:
source = textwrap.dedent('''
import os

def run(cmd):
    os.system(cmd)

run(input())
''')
print(source)

In [ ]:
taint = taint_to_dict(source)
capability = capability_to_dict(source, name='nb_example.py')
resource = resource_to_dict(source, name='nb_example.py')

decision = SafetyDecisionEngine(policies=[ExternalAccessPolicy()]).evaluate(
    taint_results=taint,
    capability_results=capability,
    resource_results=resource,
)

print('verdict:', decision.verdict)
print('reasons:', decision.reasons)
print('taint findings:', taint['total'])
print('capabilities:', capability['summary']['capabilities'])
print('resource risk:', resource['risk_score'])

In [ ]:
manifest = generate_manifest_from_source(
    source,
    source_name='nb_example.py',
    timestamp='2026-05-19T00:00:00Z',
    analysis_version='2026.05',
)
verification = ManifestVerificationEngine().verify(manifest=manifest)

cfg = build_cfg(source=source)
ir = build_safety_ir_from_source(
    source,
    source_name='nb_example.py',
    timestamp='2026-05-19T00:00:00Z',
    analysis_version='2026.05',
)

print('manifest trusted:', verification.trusted)
print('cfg blocks:', cfg.to_dict()['summary']['total_blocks'])
print('ir node_type:', ir.to_dict()['node_type'])